# Power BI Reporting Dataset Preparation

This notebook prepares clean reporting tables for Power BI.

The objective is not to repeat previous analyses, but to create a structured reporting layer that Power BI can read directly from:

`outputs/powerbi_operational/`

The notebook creates:
- dimension tables
- fact tables
- KPI summary tables
- dashboard-ready CSV outputs

In [13]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../outputs/powerbi_operational")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

part_master = pd.read_csv(DATA_DIR / "part_master.csv")
inventory_status = pd.read_csv(DATA_DIR / "inventory_status.csv")
supplier_master = pd.read_csv(DATA_DIR / "supplier_master.csv")
equipment_models = pd.read_csv(DATA_DIR / "equipment_models.csv")
demand_history = pd.read_csv(DATA_DIR / "demand_history_weekly.csv")
purchase_requisitions = pd.read_csv(DATA_DIR / "purchase_requisitions.csv")

demand_history["Week_Start"] = pd.to_datetime(demand_history["Week_Start"])

## Dimension Tables

Dimension tables contain descriptive attributes used for filtering, slicing, and grouping in Power BI.

In [14]:
dim_parts = part_master[
    [
        "Part_ID",
        "Part_Name",
        "Equipment_Family",
        "Equipment_Model",
        "Part_Category",
        "Demand_Profile",
        "Criticality",
        "Supplier_ID",
        "Supplier_Region",
        "Unit_Cost_EUR",
        "Storage_Type",
        "Lifecycle_Status",
        "ABC_Class",
        "Movement_Class",
        "Manual_Location",
        "UDC_Type",
        "Dim_X_mm",
        "Dim_Y_mm",
        "Dim_Z_mm",
        "Unit_Volume_cm3",
        "Max_Dimension_mm",
        "Stock_Managed"
    ]
].copy()

dim_suppliers = supplier_master.copy()
dim_equipment = equipment_models.copy()

dim_date = (
    demand_history[["Week_Start"]]
    .drop_duplicates()
    .sort_values("Week_Start")
    .copy()
)

dim_date["Year"] = dim_date["Week_Start"].dt.year
dim_date["Month"] = dim_date["Week_Start"].dt.month
dim_date["Quarter"] = dim_date["Week_Start"].dt.quarter
dim_date["Year_Week"] = (
    dim_date["Week_Start"].dt.isocalendar().year.astype(str) +
    "-W" +
    dim_date["Week_Start"].dt.isocalendar().week.astype(str).str.zfill(2)
)

## Fact Tables

Fact tables contain measurable operational activity such as demand history, inventory position, and purchase requisitions.

In [15]:
fact_weekly_demand = demand_history.copy()

fact_inventory_status = (
    inventory_status
    .merge(
        part_master[
            [
                "Part_ID",
                "Unit_Cost_EUR",
                "ABC_Class",
                "Movement_Class",
                "Movement_Lines_36M",
                "Criticality",
                "Supplier_ID",
                "Supplier_Region",
                "Part_Category",
                "Unit_Volume_cm3",
                "Stock_Qty",
                "Max_Stock_Qty",
                "Stock_Managed"
            ]
        ],
        on="Part_ID",
        how="left"
    )
)

fact_inventory_status["Estimated_Inventory_Value_EUR"] = (
    fact_inventory_status["Available_Qty"] *
    fact_inventory_status["Unit_Cost_EUR"]
)

fact_inventory_status["Below_Reorder_Point"] = (
    fact_inventory_status["Available_Qty"] <
    fact_inventory_status["Reorder_Point_Qty"]
)

fact_inventory_status["Stock_Coverage_Weeks"] = np.where(
    fact_inventory_status["Avg_Weekly_Demand"] > 0,
    fact_inventory_status["Available_Qty"] /
    fact_inventory_status["Avg_Weekly_Demand"],
    np.nan
)

fact_inventory_status["Total_Occupied_Volume_cm3"] = (
    fact_inventory_status["Stock_Qty"] *
    fact_inventory_status["Unit_Volume_cm3"]
)

fact_purchase_requisitions = (
    purchase_requisitions
    .merge(
        part_master[
            [
                "Part_ID",
                "Part_Name",
                "Part_Category",
                "ABC_Class",
                "Movement_Class",
                "Criticality",
                "Unit_Cost_EUR",
                "Supplier_Region"
            ]
        ],
        on="Part_ID",
        how="left"
    )
)

## Dashboard KPI Tables

These tables provide compact KPI outputs for Power BI cards, summaries, and dashboard overview pages.

In [ ]:
kpi_summary = pd.DataFrame({
    "KPI": [
        "Total parts",
        "Stock-managed parts",
        "Total available quantity",
        "Estimated inventory value EUR",
        "Parts below reorder point",
        "Planner exceptions",
        "Purchase requisitions",
        "Purchase requisition value EUR",
        "Critical parts below reorder point"
    ],
    "Value": [
        fact_inventory_status["Part_ID"].nunique(),
        fact_inventory_status["Stock_Managed"].sum(),
        fact_inventory_status["Available_Qty"].sum(),
        round(fact_inventory_status["Estimated_Inventory_Value_EUR"].sum(), 2),
        fact_inventory_status["Below_Reorder_Point"].sum(),
        fact_inventory_status["Planner_Exception_Flag"].sum(),
        fact_purchase_requisitions["Purchase_Requisition_ID"].nunique(),
        round(fact_purchase_requisitions["Estimated_Value_EUR"].sum(), 2),
        fact_inventory_status[
            (fact_inventory_status["Criticality"] == "Critical") &
            (fact_inventory_status["Below_Reorder_Point"])
        ]["Part_ID"].nunique()
    ]
})

inventory_value_by_abc = (
    fact_inventory_status
    .groupby("ABC_Class")
    .agg(
        Parts=("Part_ID", "count"),
        Estimated_Inventory_Value_EUR=("Estimated_Inventory_Value_EUR", "sum")
    )
    .round(2)
    .reset_index()
)

movement_summary = (
    fact_inventory_status
    .groupby("Movement_Class")
    .agg(
        Parts=("Part_ID", "count"),
        Total_Movement_Lines=("Movement_Lines_36M", "sum"),
        Parts_Below_Reorder=("Below_Reorder_Point", "sum"),
        Estimated_Inventory_Value_EUR=("Estimated_Inventory_Value_EUR", "sum")
    )
    .round(2)
    .reset_index()
)

supplier_replenishment_summary = (
    fact_inventory_status
    .groupby("Supplier_Region")
    .agg(
        Parts=("Part_ID", "count"),
        Parts_Below_Reorder=("Below_Reorder_Point", "sum"),
        Planner_Exceptions=("Planner_Exception_Flag", "sum"),
        Suggested_Replenishment_Qty=("Suggested_Replenishment_Qty", "sum")
    )
    .round(2)
    .reset_index()
)

## Export Power BI Reporting Tables

The tables are exported as CSV files into `outputs/powerbi/`.

Power BI can connect directly to this folder.  
After the dashboard is built once, refreshing Power BI will update the visuals from these files.

In [ ]:
dim_parts.to_csv(OUTPUT_DIR / "dim_parts.csv", index=False)
dim_suppliers.to_csv(OUTPUT_DIR / "dim_suppliers.csv", index=False)
dim_equipment.to_csv(OUTPUT_DIR / "dim_equipment.csv", index=False)
dim_date.to_csv(OUTPUT_DIR / "dim_date.csv", index=False)

fact_weekly_demand.to_csv(OUTPUT_DIR / "fact_weekly_demand.csv", index=False)
fact_inventory_status.to_csv(OUTPUT_DIR / "fact_inventory_status.csv", index=False)
fact_purchase_requisitions.to_csv(OUTPUT_DIR / "fact_purchase_requisitions.csv", index=False)

kpi_summary.to_csv(OUTPUT_DIR / "kpi_summary.csv", index=False)
inventory_value_by_abc.to_csv(OUTPUT_DIR / "inventory_value_by_abc.csv", index=False)
movement_summary.to_csv(OUTPUT_DIR / "movement_summary.csv", index=False)
supplier_replenishment_summary.to_csv(OUTPUT_DIR / "supplier_replenishment_summary.csv", index=False)

print("Power BI reporting tables exported to:", OUTPUT_DIR)

Power BI reporting tables exported to: ..\outputs\powerbi_operational


## Output Summary

This notebook creates the reporting layer used by Power BI.

The exported files are not a separate analysis.  
They are cleaned and structured outputs designed for dashboard development, refresh, and operational reporting.

In [ ]:
exported_files = sorted([file.name for file in OUTPUT_DIR.glob("*.csv")])

pd.DataFrame({
    "Exported_File": exported_files
})

,Exported_File
0,dim_date.csv
1,dim_equipment.csv
2,dim_parts.csv
3,dim_suppliers.csv
4,fact_inventory_status.csv
5,fact_purchase_requisitions.csv
6,fact_weekly_demand.csv
7,inventory_value_by_abc.csv
8,kpi_summary.csv
9,movement_summary.csv
